# STEPS FOR KNN 
1. ## import data and indicate x-y values
2. ## split the data into train and test datasets
3. ## set cross validated KNN model object and fit it to train dataset
4. ## choose best "k" for tuned KNN model and find final RMSE

# 1.Import data

In [1]:
import pandas as pd
import numpy as np
hitters=pd.read_csv("../input/hittlers/Hitters.csv")
df=hitters.copy()
df.columns


Index(['AtBat', 'Hits', 'HmRun', 'Runs', 'RBI', 'Walks', 'Years', 'CAtBat',
       'CHits', 'CHmRun', 'CRuns', 'CRBI', 'CWalks', 'League', 'Division',
       'PutOuts', 'Assists', 'Errors', 'Salary', 'NewLeague'],
      dtype='object')

In [2]:
df.head(2)

,AtBat,Hits,HmRun,Runs,RBI,Walks,Years,CAtBat,CHits,CHmRun,CRuns,CRBI,CWalks,League,Division,PutOuts,Assists,Errors,Salary,NewLeague
0,293,66,1,30,29,14,1,293,66,1,30,29,14,A,E,446,33,20,NaN,A
1,315,81,7,24,38,39,14,3449,835,69,321,414,375,N,W,632,43,10,475.0,N


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 322 entries, 0 to 321
Data columns (total 20 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   AtBat      322 non-null    int64  
 1   Hits       322 non-null    int64  
 2   HmRun      322 non-null    int64  
 3   Runs       322 non-null    int64  
 4   RBI        322 non-null    int64  
 5   Walks      322 non-null    int64  
 6   Years      322 non-null    int64  
 7   CAtBat     322 non-null    int64  
 8   CHits      322 non-null    int64  
 9   CHmRun     322 non-null    int64  
 10  CRuns      322 non-null    int64  
 11  CRBI       322 non-null    int64  
 12  CWalks     322 non-null    int64  
 13  League     322 non-null    object 
 14  Division   322 non-null    object 
 15  PutOuts    322 non-null    int64  
 16  Assists    322 non-null    int64  
 17  Errors     322 non-null    int64  
 18  Salary     263 non-null    float64
 19  NewLeague  322 non-null    object 
dtypes: float64

In [4]:
df.shape

(322, 20)



####Create X variables( from dummies ) and y variable




In [5]:
import pandas as pd
import numpy as np
df=df.dropna()
df.shape

(263, 20)

In [6]:
dms=pd.get_dummies(df[['League', 'Division', 'NewLeague']])  
dms.head(2)

,League_A,League_N,Division_E,Division_W,NewLeague_A,NewLeague_N
1,0,1,0,1,0,1
2,1,0,0,1,1,0


In [7]:
x=df.drop(["Salary",'League', 'Division', 'NewLeague'], axis=1).astype("float64")
x.head(2)


,AtBat,Hits,HmRun,Runs,RBI,Walks,Years,CAtBat,CHits,CHmRun,CRuns,CRBI,CWalks,PutOuts,Assists,Errors
1,315.0,81.0,7.0,24.0,38.0,39.0,14.0,3449.0,835.0,69.0,321.0,414.0,375.0,632.0,43.0,10.0
2,479.0,130.0,18.0,66.0,72.0,76.0,3.0,1624.0,457.0,63.0,224.0,266.0,263.0,880.0,82.0,14.0


In [8]:
X = pd.concat([x, dms[['League_N', 'Division_W', 'NewLeague_N']]], axis=1)
X.head(2)

,AtBat,Hits,HmRun,Runs,RBI,Walks,Years,CAtBat,CHits,CHmRun,CRuns,CRBI,CWalks,PutOuts,Assists,Errors,League_N,Division_W,NewLeague_N
1,315.0,81.0,7.0,24.0,38.0,39.0,14.0,3449.0,835.0,69.0,321.0,414.0,375.0,632.0,43.0,10.0,1,1,1
2,479.0,130.0,18.0,66.0,72.0,76.0,3.0,1624.0,457.0,63.0,224.0,266.0,263.0,880.0,82.0,14.0,0,1,0


In [9]:
y=df["Salary"]

 # 2.Split the data into train and test data

In [10]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train, y_test= train_test_split(X,y,test_size=0.25, random_state=42)

# 3.set cross validated KNN model object and fit it to train dataset

In [11]:

from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsRegressor

knn=KNeighborsRegressor()
k_params={"n_neighbors":np.arange(1,30,1)}


knncv=GridSearchCV(knn,k_params, cv=10)
knncv.fit(X_train, y_train)


GridSearchCV(cv=10, error_score=nan,
             estimator=KNeighborsRegressor(algorithm='auto', leaf_size=30,
                                           metric='minkowski',
                                           metric_params=None, n_jobs=None,
                                           n_neighbors=5, p=2,
                                           weights='uniform'),
             iid='deprecated', n_jobs=None,
             param_grid={'n_neighbors': array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29])},
             pre_dispatch='2*n_jobs', refit=True, return_train_score=False,
             scoring=None, verbose=0)

 # 4.choose best "k" for tuned KNN model and find final RMSE for test error

In [12]:
knncv.best_params_["n_neighbors"]   #the best k for this dataset

8

In [13]:
from sklearn.metrics import mean_squared_error, r2_score 
knn_tune=KNeighborsRegressor(n_neighbors = knncv.best_params_["n_neighbors"]).fit(X_train, y_train)
np.sqrt(mean_squared_error(y_test, knn_tune.predict(X_test)))

413.7094731463598

# CROSSCHECK and exercises for different k values with not validated model

#### Let's create not validated KNN model object and fit it WİTH default n_neighbors=5  (this is our k number for now) and find RMSE




In [14]:
# Create KNN model object with default settings (n_neighbors=5 which means k value=5)
from sklearn.neighbors import KNeighborsRegressor
knn_model=KNeighborsRegressor().fit(X_train, y_train)
knn_model


KNeighborsRegressor(algorithm='auto', leaf_size=30, metric='minkowski',
                    metric_params=None, n_jobs=None, n_neighbors=5, p=2,
                    weights='uniform')

In [15]:
knn_model.n_neighbors

5

In [16]:
#Find Test Error (RMSE) within predicted y values and "test data real y values". 
#To do that, first predict y values from "test data x values". 
#In brief,from test data x values--> gain predicted y values and compare it with test data real y values
ypredict=knn_model.predict(X_test)
ypredict[0:5]

array([ 510.3334,  808.3334,  772.5   ,  125.5   , 1005.    ])

In [17]:
#### Find Test Error (RMSE) within predicted y values and "test data y values.
from sklearn.metrics import mean_squared_error, r2_score 
np.sqrt(mean_squared_error(ypredict,y_test))

426.6570764525201

# CROSSCHECK and exercises for different k values with not validated and validated model

In [18]:
from sklearn.model_selection import train_test_split, GridSearchCV,cross_val_score
RMSE=[]     
RMSECV=[]    #rmse values with cross validation.
for k in range(10):
    k=k+1
    knn_model = KNeighborsRegressor(n_neighbors = k).fit(X_train, y_train)    #train error
    ytrainpredict=knn_model.predict(X_train)
    rmse=np.sqrt(mean_squared_error(ytrainpredict,y_train))
    rmsecv=np.sqrt(-1*cross_val_score(knn_model,X_train,y_train,cv=10, scoring="neg_mean_squared_error").mean())
    RMSE.append(rmse)
    RMSECV.append(rmsecv)
    print("for k=", k, "    RMSE value=",rmse, "     Cross Validated RMSE", rmsecv)
    #Here the result of RMSE with CV : the range of it is btw 283 and 301. It is smaller than rmse range.

for k= 1     RMSE value= 0.0      Cross Validated RMSE 325.39475147063825
for k= 2     RMSE value= 179.52761335480352      Cross Validated RMSE 293.24000183333817
for k= 3     RMSE value= 205.20157172291863      Cross Validated RMSE 283.7486667487823
for k= 4     RMSE value= 220.5139794876305      Cross Validated RMSE 286.3240222024089
for k= 5     RMSE value= 239.64671325413764      Cross Validated RMSE 290.0705466132226
for k= 6     RMSE value= 243.5904190007242      Cross Validated RMSE 298.1263115575851
for k= 7     RMSE value= 258.1478781634636      Cross Validated RMSE 294.77070479194987
for k= 8     RMSE value= 266.05374203349805      Cross Validated RMSE 291.98672028891235
for k= 9     RMSE value= 269.73782093553376      Cross Validated RMSE 295.7162739573105
for k= 10     RMSE value= 271.2798300436963      Cross Validated RMSE 301.31047022701154


### Now, we gain train error(RMSE):                   from train data
### And we gain train error(Cross Validated RMSE):    from cross validated model

## Note that,it is not important to have small RMSE which is misleading here.
## We are looking for small range. RMSE range is 179-271. CV RMSE range is 283-301.
## We always consider that test error must be calculated from validated  model.

In [19]:
knn_tune=KNeighborsRegressor(n_neighbors = knncv.best_params_["n_neighbors"]).fit(X_train, y_train)
np.sqrt(mean_squared_error(y_test, knn_tune.predict(X_test)))

413.7094731463598

# 413 is real RMSE for test data

#### https://github.com/mvahit/DSMLBC/blob/master/README.md